# 05 — Classification and extent change

Transfer NAIP labels to the 30 m grid, train a classifier on the phenology cube, and map gallery extent at a handful of benchmark epochs.

**Reads** labels from `03`, composites from `04`  
**Writes** per-epoch class rasters, area-adjusted accuracy table  
**Status** Phase 4 — skeleton

> Skeleton. Section headings and the config cell are in place; the analysis cells are deliberately empty for the group to fill in together.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
from sklearn.ensemble import RandomForestClassifier

def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

print("Imports OK")
print(f"  repo : {REPO}")

## 1. Configuration

Every parameter lives here. Pointing this notebook at another tile or another river is a single-cell edit.

In [ ]:
# ---- The pilot tile ----
TILE = "13TFJ"                       # holds Angostura, Buffalo Gap, Red Shirt, Scenic

# ---- Example reaches: one window per 8-digit USGS gauge inside 13TFJ ----
# Gauge-anchored so every window has a flow record to read alongside it (notebook 07).
# These are the walkthrough reaches, NOT the full corridor -- scaling is Phase 6.
EXAMPLE_WINDOWS = [
    {"site_no": "06401500", "name": "Angostura",   "lon": -103.4340, "lat": 43.3470},
    {"site_no": "06402600", "name": "Buffalo Gap", "lon": -103.2350, "lat": 43.4230},
    {"site_no": "06403700", "name": "Red Shirt",   "lon": -102.8921, "lat": 43.6724},
    {"site_no": "06408650", "name": "Scenic",      "lon": -102.5500, "lat": 43.7800},
]
WINDOW_HALF_M = 1000                 # half-width -> 2 x 2 km windows, as in notebook 03
# VERIFY: lon/lat for all but Red Shirt are approximate -- replace from the
# usgs_gauges layer of cheyenne_corridor_aoi.gpkg on first run.

# ---- Upstream runs ----
LABEL_RUN     = "labels_smoketest_redshirt_06403700_2022"
PHENOLOGY_RUN = "phenology_vbet_13TFJ"

# ---- Label transfer ----
# NAIP is 60 cm, Landsat is 30 m. A 30 m cell is ~2,500 NAIP pixels; only cells that are
# overwhelmingly one class make honest training data.
PURITY_MIN = 0.70                    # fraction of the 30 m cell that must be one class

# ---- Benchmark epochs ----
# Hard classification at a few dates, NOT an annual series -- pushing a 2012-2022-trained
# classifier back to 1984 carries real domain shift (TM vs ETM+ vs OLI band-pass).
BENCHMARK_YEARS = [1985, 1995, 2005, 2015, 2022]

# ---- Classifier ----
MODEL_KIND = "random_forest"
N_TREES    = 300
SEED       = 42

# ---- Outputs ----
RUN_NAME = f"extent_{PHENOLOGY_RUN}"
FIG_SUBDIR = RUN_NAME               # figures/<run>/
OUT_DIR  = DATA_DIR / RUN_NAME

## 2. Aggregate labels to 30 m

Majority class per 30 m cell, keeping only cells above `PURITY_MIN`. Report how many labels survive — the purity filter biases training toward wider galleries, and that bias belongs in the write-up.

## 3. Train

Fit on the NAIP-epoch composite. Hold out spatially, not randomly: neighbouring pixels are not independent and a random split will flatter the accuracy.

## 4. Classify the benchmark epochs

Apply to each year in `BENCHMARK_YEARS`.

## 5. Area-adjusted accuracy

Olofsson-style estimator with confidence intervals. Raw pixel counts are biased; report adjusted area or don't report area.

## 6. Extent change

Gallery area per epoch with its uncertainty band. **State plainly** that this product is coarse in time — that is the honest counterpart to notebook 06.

## 7. Save and record the run

Every output gets a manifest in `runs/` — small, text, always committed, even when the raster it describes is not.

In [ ]:
manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "05_Classification_and_Extent_Change.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "inputs":      {},          # STAC item IDs, upstream run names, source manifests
    "parameters":  {},          # everything from the config cell
    "environment": {"python": sys.version.split()[0]},
    "results":     {},
    "outputs":     [],
}

# manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
# manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")

## What comes next

Gate: is the change signal larger than its uncertainty? Notebook 06 answers the same question a second, more sensor-robust way.